In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!nvidia-smi
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

Mon Mar  9 06:02:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -U transformers peft accelerate sentencepiece
!pip install llama-cpp-python -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 35.3 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 20.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00


In [4]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00


In [5]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
LORA_PATH = "/content/drive/MyDrive/adapters"
MERGED_PATH = "/content/merged_model"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, LORA_PATH)

# merge LoRA into base model
model = model.merge_and_unload()

# save merged model
model.save_pretrained(MERGED_PATH)

# save tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained(MERGED_PATH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

('/content/merged_model/tokenizer_config.json',
 '/content/merged_model/chat_template.jinja',
 '/content/merged_model/tokenizer.json')

In [6]:
!ls /content/drive/MyDrive/adapters


adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json


In [7]:
!git clone https://github.com/ggerganov/llama.cpp


Cloning into 'llama.cpp'...
remote: Enumerating objects: 82612, done.
remote: Counting objects: 100% (191/191), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 82612 (delta 100), reused 30 (delta 29), pack-reused 82421 (from 4)
Receiving objects: 100% (82612/82612), 313.26 MiB | 17.72 MiB/s, done.
Resolving deltas: 100% (59416/59416), done.
Updating files: 100% (2435/2435), done.


In [8]:
%cd ./llama.cpp


/content/llama.cpp


In [9]:
# !cmake -B build
# !cmake --build build --config Release -j
# cmake --build . --parallel 1

!cmake -B build
!cmake --build build --parallel 1


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [10]:
cd /content/llama.cpp


/content/llama.cpp


In [11]:
!ls /content


drive  llama.cpp  merged_model	sample_data


In [12]:
%cd /content/llama.cpp

!python convert_hf_to_gguf.py /content/merged_model \
  --outfile model-f16.gguf \
  --outtype f16

/content/llama.cpp
INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> F16, shape = {2048, 256}
INFO:hf-t

In [13]:
!ls


AGENTS.md		      convert_llama_ggml_to_gguf.py  models
AUTHORS			      convert_lora_to_gguf.py	     mypy.ini
benches			      docs			     pocs
build			      examples			     poetry.lock
build-xcframework.sh	      flake.lock		     pyproject.toml
ci			      flake.nix			     pyrightconfig.json
CLAUDE.md		      ggml			     README.md
cmake			      gguf-py			     requirements
CMakeLists.txt		      grammars			     requirements.txt
CMakePresets.json	      include			     scripts
CODEOWNERS		      LICENSE			     SECURITY.md
common			      licenses			     src
CONTRIBUTING.md		      Makefile			     tests
convert_hf_to_gguf.py	      media			     tools
convert_hf_to_gguf_update.py  model-f16.gguf		     vendor


In [14]:
%cd /content/llama.cpp

/content/llama.cpp


In [15]:
!ls | grep convert

convert_hf_to_gguf.py
convert_hf_to_gguf_update.py
convert_llama_ggml_to_gguf.py
convert_lora_to_gguf.py


In [16]:
!python convert_hf_to_gguf.py /content/merged_model \
  --outfile model-f16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.bfloat16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> F16, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.bfloat16 --> F16, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_o

In [17]:
!ls /content/llama.cpp/build


bin		       CTestTestfile.cmake    llama.pc		   tests
CMakeCache.txt	       DartConfiguration.tcl  llama-version.cmake  tools
CMakeFiles	       examples		      Makefile		   vendor
cmake_install.cmake    ggml		      pocs
common		       license.cpp	      src
compile_commands.json  llama-config.cmake     Testing


In [18]:
!ls /content/llama.cpp/build/bin


libggml-base.so		       llama-qwen2vl-cli
libggml-base.so.0	       llama-results
libggml-base.so.0.9.7	       llama-retrieval
libggml-cpu.so		       llama-save-load-state
libggml-cpu.so.0	       llama-server
libggml-cpu.so.0.9.7	       llama-simple
libggml.so		       llama-simple-chat
libggml.so.0		       llama-speculative
libggml.so.0.9.7	       llama-speculative-simple
libllama.so		       llama-template-analysis
libllama.so.0		       llama-tokenize
libllama.so.0.0.8248	       llama-tts
libmtmd.so		       llama-vdot
libmtmd.so.0		       test-alloc
libmtmd.so.0.0.8248	       test-arg-parser
llama-batched		       test-autorelease
llama-batched-bench	       test-backend-ops
llama-bench		       test-backend-sampler
llama-cli		       test-barrier
llama-completion	       test-c
llama-convert-llama2c-to-ggml  test-chat
llama-cvector-generator        test-chat-auto-parser
llama-debug		       test-chat-peg-parser
llama-debug-template-parser    test-chat-template
llama-diffusion-cli	       test

In [19]:
ls

AGENTS.md                       grammars/
AUTHORS                         include/
benches/                        LICENSE
build/                          licenses/
build-xcframework.sh*           Makefile
ci/                             media/
CLAUDE.md                       model-f16.gguf
cmake/                          models/
CMakeLists.txt                  mypy.ini
CMakePresets.json               pocs/
CODEOWNERS                      poetry.lock
common/                         pyproject.toml
CONTRIBUTING.md                 pyrightconfig.json
convert_hf_to_gguf.py*          README.md
convert_hf_to_gguf_update.py*   requirements/
convert_llama_ggml_to_gguf.py*  requirements.txt
convert_lora_to_gguf.py*        scripts/
docs/                           SECURITY.md
examples/                       src/
flake.lock                      tests/
flake.nix                       tools/
ggml/                           vendor/
gguf-py/


In [24]:
import os
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "/content/merged_model"
OUT_DIR = "/content/quantized"
os.makedirs(OUT_DIR, exist_ok=True)

def quantize(bits):
    print(f"\n--- INT{bits} Quantization ---")

    if bits == 8:
        bnb = BitsAndBytesConfig(load_in_8bit=True)
    else:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        quantization_config=bnb
    )

    save_path = f"{OUT_DIR}/model-int{bits}"
    model.save_pretrained(save_path)

    mem = model.get_memory_footprint() / 1024**2
    print(f"Memory footprint: {mem:.2f} MB")

    del model
    torch.cuda.empty_cache()

quantize(8)
quantize(4)


--- INT8 Quantization ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Memory footprint: 1174.18 MB

--- INT4 Quantization ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Memory footprint: 712.18 MB


In [25]:
!ls -lh ./model-f16.gguf


-rw-r--r-- 1 root root 2.1G Mar  9 06:33 ./model-f16.gguf


In [28]:
%cd /content/llama.cpp

/content/llama.cpp


In [29]:
!./build/bin/llama-quantize model-f16.gguf /content/quantized/model-q8_0.gguf q8_0

main: build = 8248 (5f4cdac38)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'model-f16.gguf' to '/content/quantized/model-q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 31 key-value pairs and 201 tensors from model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6: 

In [30]:
!./build/bin/llama-quantize model-f16.gguf /content/quantized/model-q4_0.gguf q4_0

main: build = 8248 (5f4cdac38)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing 'model-f16.gguf' to '/content/quantized/model-q4_0.gguf' as Q4_0
llama_model_loader: loaded meta data with 31 key-value pairs and 201 tensors from model-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_loader: - kv   5:                       llama.context_length u32              = 2048
llama_model_loader: - kv   6: 

In [31]:
!ls -lh /content/quantized

total 1.7G
drwxr-xr-x 2 root root 4.0K Mar  9 06:39 model-int4
drwxr-xr-x 2 root root 4.0K Mar  9 06:38 model-int8
-rw-r--r-- 1 root root 608M Mar  9 06:46 model-q4_0.gguf
-rw-r--r-- 1 root root 1.1G Mar  9 06:45 model-q8_0.gguf


In [32]:
import os

files = {
    "FP16 GGUF": "/content/llama.cpp/model-f16.gguf",
    "INT8 GGUF": "/content/quantized/model-q8_0.gguf",
    "INT4 GGUF": "/content/quantized/model-q4_0.gguf"
}

print(f"{'Format':<15} | {'Size (MB)':>10}")
print("-" * 30)

for name, path in files.items():
    size = os.path.getsize(path) / 1024**2
    print(f"{name:<15} | {size:>10.2f}")

Format          |  Size (MB)
------------------------------
FP16 GGUF       |    2099.05
INT8 GGUF       |    1115.62
INT4 GGUF       |     607.23


In [35]:
from llama_cpp import Llama

llm = Llama(
    model_path="/content/quantized/model-q4_0.gguf",
    n_ctx=2048,
    chat_format="llama-3",
    repetition_penalty=1.2,
    verbose=False
)

response = llm.create_chat_completion(
    messages=[
    {"role": "system", "content": "Analyze the HR scenario step by step, clearly explain the HR reasoning, and conclude."},
    {"role": "user", "content": "An experienced employee has received an external job offer with a higher salary. As an HR manager, what strategies could be used to retain this employee without creating pay inequality within the team?"}
    ],
    temperature=0.1,
    max_tokens=300
)

print(response["choices"][0]["message"]["content"])

HR reasoning:

HR strategies:

1. Communicate the benefits of the external offer clearly and transparently.

2. Offer a competitive salary increase to retain the employee.

3. Provide a clear path for career advancement within the organization.

4. Offer flexible work arrangements to accommodate the external offer.

5. Provide a clear explanation of the salary comparison and benefits comparison.

6. Communicate the company's commitment to equal pay for equal work.

7. Provide training and development opportunities to ensure the employee's success in the new role.

8. Communicate the company's commitment to maintaining a diverse workforce.

HR reasoning:

HR reasoning:

HR strategies:

1. Communicate the benefits of the external offer clearly and transparently.

2. Offer a competitive salary increase to retain the employee.

3. Provide a clear path for career advancement within the organization.

4. Offer flexible work arrangements to accommodate the external offer.

5. Provide a clear 

In [34]:
import os

models = {
    "BF16": "/content/merged_model",
    "INT8 (bnb)": "/content/quantized/model-int8",
    "INT4 (bnb)": "/content/quantized/model-int4",
    "GGUF f16": "/content/llama.cpp/model-f16.gguf",
    "GGUF q8_0": "/content/quantized/model-q8_0.gguf",
    "GGUF q4_0": "/content/quantized/model-q4_0.gguf",
}

def get_size_mb(path):
    if os.path.isfile(path):
        return os.path.getsize(path) / 1024**2
    total = 0
    for root, _, files in os.walk(path):
        for f in files:
            total += os.path.getsize(os.path.join(root, f))
    return total / 1024**2

print(f"{'Format':<15} | {'Size (MB)':>10}")
print("-" * 30)

for name, path in models.items():
    print(f"{name:<15} | {get_size_mb(path):>10.2f}")

Format          |  Size (MB)
------------------------------
BF16            |    2101.65
INT8 (bnb)      |    1175.73
INT4 (bnb)      |     727.13
GGUF f16        |    2099.05
GGUF q8_0       |    1115.62
GGUF q4_0       |     607.23


In [36]:
from llama_cpp import Llama
import time

def benchmark_gguf(path, label, prompt):
    llm = Llama(
        model_path=path,
        n_ctx=2048,
        n_threads=8,
        verbose=False
    )

    start = time.time()
    llm(prompt, max_tokens=128, echo=False)
    end = time.time()

    tps = 128 / (end - start)
    print(f"{label}: {tps:.2f} tokens/sec")


prompt = "Explain the role of HR in improving employee engagement."

benchmark_gguf("/content/quantized/model-q8_0.gguf", "GGUF q8_0", prompt)
benchmark_gguf("/content/quantized/model-q4_0.gguf", "GGUF q4_0", prompt)
benchmark_gguf("/content/llama.cpp/model-f16.gguf", "GGUF f16", prompt)

GGUF q8_0: 180.18 tokens/sec
GGUF q4_0: 7.26 tokens/sec
GGUF f16: 44.26 tokens/sec


In [38]:
import time
import pandas as pd
from transformers import pipeline
from llama_cpp import Llama

PROMPT = """
### Instruction:
Explain the importance of employee engagement in an organization.

### Input:

### Response:
"""

results = []


# FP16 / Base Model
start = time.time()

pipe_fp16 = pipeline(
    "text-generation",
    model="/content/merged_model",
    tokenizer="/content/merged_model",
    max_new_tokens=120
)

out = pipe_fp16(PROMPT)[0]["generated_text"]
t = time.time() - start

results.append({
    "Format": "FP16",
    "Speed(sec)": round(t,2),
    "Output": out[:200]
})


# INT8 Model
start = time.time()

pipe_int8 = pipeline(
    "text-generation",
    model="/content/quantized/model-int8",
    tokenizer="/content/merged_model",
    max_new_tokens=120
)

out = pipe_int8(PROMPT)[0]["generated_text"]
t = time.time() - start

results.append({
    "Format": "INT8",
    "Speed(sec)": round(t,2),
    "Output": out[:200]
})


# INT4 Model
start = time.time()

pipe_int4 = pipeline(
    "text-generation",
    model="/content/quantized/model-int4",
    tokenizer="/content/merged_model",
    max_new_tokens=120
)

out = pipe_int4(PROMPT)[0]["generated_text"]
t = time.time() - start

results.append({
    "Format": "INT4",
    "Speed(sec)": round(t,2),
    "Output": out[:200]
})


# GGUF Model
start = time.time()

llm = Llama(
    model_path="/content/quantized/model-q4_0.gguf",
    n_ctx=2048
)

out = llm(PROMPT, max_tokens=120)["choices"][0]["text"]
t = time.time() - start

results.append({
    "Format": "GGUF (q4_0)",
    "Speed(sec)": round(t,2),
    "Output": out[:200]
})


# Show results
df = pd.DataFrame(results)

print("Quantization Comparison")
display(df)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
llama_model_loader: loaded meta data with 31 key-value pairs and 201 tensors from /content/quantized/model-q4_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Merged_Model
llama_model_loader: - kv   3:                         general.size_label str              = 1.1B
llama_model_loader: - kv   4:                          llama.block_count u32              = 22
llama_model_lo

Quantization Comparison


,Format,Speed(sec),Output
0,FP16,15.94,\n### Instruction:\nExplain the importance of ...
1,INT8,20.07,\n### Instruction:\nExplain the importance of ...
2,INT4,11.58,\n### Instruction:\nExplain the importance of ...
3,GGUF (q4_0),15.71,Employee engagement is the level of commitment...


In [39]:
!zip -r /content/week8_models.zip \
/content/merged_model \
/content/quantized \
/content/llama.cpp/model-f16.gguf

  adding: content/merged_model/ (stored 0%)
  adding: content/merged_model/model.safetensors (deflated 21%)
  adding: content/merged_model/tokenizer_config.json (deflated 44%)
  adding: content/merged_model/tokenizer.json (deflated 85%)
  adding: content/merged_model/generation_config.json (deflated 29%)
  adding: content/merged_model/config.json (deflated 49%)
  adding: content/merged_model/chat_template.jinja (deflated 60%)
  adding: content/quantized/ (stored 0%)
  adding: content/quantized/model-q8_0.gguf (deflated 4%)
  adding: content/quantized/model-int4/ (stored 0%)
  adding: content/quantized/model-int4/model.safetensors (deflated 9%)
  adding: content/quantized/model-int4/generation_config.json (deflated 29%)
  adding: content/quantized/model-int4/config.json (deflated 56%)
  adding: content/quantized/model-q4_0.gguf (deflated 5%)
  adding: content/quantized/model-int8/ (stored 0%)
  adding: content/quantized/model-int8/model.safetensors (deflated 13%)
  adding: content/quant

In [41]:
from google.colab import files
files.download("/content/week8_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>